# 🧩 Aceleración con LCM-LoRA (Latent Consistency Models)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mattbarreto/ifts24-lab-pdi-2026/blob/master/009%20-%20modelos_difusion/06_Aceleracion_LCM_LoRA.ipynb)

**Procesamiento Digital de Imágenes — IFTS24**
Prof. Matias Barreto — 2026

**Unidad 9 · Bloque 6 — 60 minutos**

---

## Al terminar este bloque vas a poder:

1. Cargar un LoRA de aceleración LCM sobre un modelo base SDXL o SD 1.5.
2. Generar imágenes de calidad comparable en 4-8 pasos en lugar de 20-50.
3. Comprender el efecto de `guidance_scale` en el contexto específico de LCM.
4. Combinar múltiples LoRAs y entender los requisitos de compatibilidad con el modelo base.

---

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **LoRA** | Low-Rank Adaptation: técnica de fine-tuning que añade pesos adicionales muy pequeños a un modelo sin modificar sus pesos originales. |
| **LCM** | Latent Consistency Model: variante de difusión entrenada para producir resultados de calidad en solo 2-8 pasos de inferencia. |
| **Adapter** | Módulo adicional que se agrega a un modelo base y modifica su comportamiento sin reescribir sus parámetros originales. |
| **Checkpoint** | Archivo que contiene los pesos de un modelo entrenado; se carga directamente para iniciar inferencia sin necesidad de reentrenar. |
| **Consistency model** | Modelo que aprende a mapear directamente desde ruido a imagen en muy pocos pasos usando una función de consistencia matemática. |
| **LCMScheduler** | Scheduler específico para LCM que coordina los pasos de consistencia; reemplaza al scheduler estándar del modelo base. |

In [ ]:
import torch
import sys

print("=== Verificación de Hardware ===")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memoria GPU: {gpu_memory:.2f} GB")
    device = "cuda"

    if gpu_memory >= 6:
        modelo_recomendado = "SDXL + LCM-LoRA"
    else:
        modelo_recomendado = "SD 1.5 + LCM-LoRA"

    print(f"\nRecomendación: {modelo_recomendado}")
else:
    print("\nSin GPU detectada")
    device = "cpu"
    print("Recomendación: SD 1.5 + LCM-LoRA en CPU (más lento)")

print(f"\nDispositivo a usar: {device}")

---

## 2. Instalación de Dependencias

**Importante:** Necesitamos diffusers actualizado con soporte para LoRA.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Instalando dependencias en Google Colab...")
    !pip install -q -U diffusers[torch] transformers accelerate safetensors
    print("✅ Instalación completada")
else:
    print("Entorno local. Asegúrate de tener diffusers >= 0.28.0")
    print("Comando: pip install -U diffusers transformers")

---

## 3. Importación de Bibliotecas

In [ ]:
import torch
from diffusers import DiffusionPipeline, LCMScheduler
from PIL import Image
import matplotlib.pyplot as plt
import time
from datetime import datetime
import os

print("✅ Bibliotecas importadas correctamente")

---

## 4. Carga del Modelo Base + LCM-LoRA

Vas a cargar SDXL y luego aplicarle el LoRA de aceleración.

In [ ]:
# Configuración
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
lcm_lora_id = "latent-consistency/lcm-lora-sdxl"

print(f"Cargando modelo base: {model_id}")
print("Esto puede tardar algunos minutos la primera vez...\n")

# Cargar pipeline base
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None,
)

pipe = pipe.to(device)

print("Modelo base cargado.")
print(f"\nCargando LCM-LoRA: {lcm_lora_id}")

# Cargar LCM-LoRA
pipe.load_lora_weights(lcm_lora_id)

# Cambiar scheduler a LCM
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

print("LCM-LoRA cargado y configurado.")

# Optimizaciones de memoria
if device == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    print("Optimizaciones de memoria aplicadas.")

---

## 5. Comparación: Con vs Sin LCM-LoRA

Vas a comparar la velocidad y calidad con diferentes números de pasos.

In [ ]:
# Prompt de prueba
prompt = "A beautiful landscape with mountains and a lake, sunset, highly detailed"

print(f"Generando con diferentes números de pasos...")
print(f"Prompt: {prompt}\n")

# Diferentes configuraciones a probar
configs = [
    (4, "4 pasos (LCM)"),
    (8, "8 pasos (LCM)"),
]

fig, axes = plt.subplots(1, len(configs), figsize=(16, 8))

for idx, (steps, label) in enumerate(configs):
    print(f"Generando: {label}")

    start_time = time.time()

    image = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=1.0,  # LCM funciona mejor con guidance bajo
        height=1024,
        width=1024,
    ).images[0]

    elapsed = time.time() - start_time

    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"{label}\nTiempo: {elapsed:.2f}s")

    print(f"  Tiempo: {elapsed:.2f}s\n")

plt.suptitle(f"Comparación de Pasos con LCM-LoRA\n{prompt}")
plt.tight_layout()
plt.show()

print("Con LCM-LoRA, 4-8 pasos suelen ser suficientes.")

---

## 6. Experimentación con Guidance Scale

LCM funciona mejor con valores de guidance_scale entre 1.0 y 2.0 (mucho menor que los tradicionales 7-15).

In [ ]:
# Probar diferentes valores de guidance
prompt_guidance = "A futuristic city with flying cars, cyberpunk style, neon lights"
guidance_values = [1.0, 1.5, 2.0]

print(f"Probando diferentes valores de guidance_scale...")
print(f"Prompt: {prompt_guidance}\n")

fig, axes = plt.subplots(1, len(guidance_values), figsize=(18, 6))

for idx, guidance in enumerate(guidance_values):
    print(f"Generando con guidance_scale={guidance}...")

    image = pipe(
        prompt=prompt_guidance,
        num_inference_steps=4,
        guidance_scale=guidance,
        height=1024,
        width=1024,
    ).images[0]

    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"Guidance: {guidance}")

plt.suptitle(f"Efecto del Guidance Scale con LCM\n{prompt_guidance}")
plt.tight_layout()
plt.show()

print("\n💡 TIP: Guidance más alto = más adherencia al prompt")
print("Pero con LCM, valores >2.0 pueden degradar la calidad")

---

## 7. Usando LCM-LoRA con SD 1.5

LCM-LoRA también funciona con Stable Diffusion 1.5, que es más liviano.

In [ ]:
# Cargar SD 1.5 con LCM-LoRA (más liviano que SDXL)
print("Cargando SD 1.5 + LCM-LoRA...\n")

pipe_sd15 = DiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

pipe_sd15 = pipe_sd15.to(device)

# Cargar LCM-LoRA para SD 1.5
pipe_sd15.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
pipe_sd15.scheduler = LCMScheduler.from_config(pipe_sd15.scheduler.config)

if device == "cuda":
    pipe_sd15.enable_attention_slicing()
    pipe_sd15.enable_vae_slicing()

print("SD 1.5 + LCM-LoRA listo.")

# Generar imagen
prompt_sd15 = "A serene japanese garden with cherry blossoms, koi pond, traditional architecture"

print(f"\nGenerando con SD 1.5...")
print(f"Prompt: {prompt_sd15}\n")

start = time.time()

image_sd15 = pipe_sd15(
    prompt=prompt_sd15,
    num_inference_steps=4,
    guidance_scale=1.0,
    height=512,
    width=512,
).images[0]

elapsed = time.time() - start

plt.figure(figsize=(10, 10))
plt.imshow(image_sd15)
plt.axis('off')
plt.title(f"SD 1.5 + LCM-LoRA (4 pasos, {elapsed:.2f}s)\n{prompt_sd15}")
plt.tight_layout()
plt.show()

print(f"\nImagen generada en {elapsed:.2f}s")
print("SD 1.5 es más rápido y usa menos memoria que SDXL")

---

## 8. Combinando Múltiples LoRAs

Podés combinar LCM-LoRA con otros LoRAs de estilo.

In [ ]:
# Ejemplo: Combinar LCM con un LoRA de estilo
# (Esto es avanzado - requiere LoRAs compatibles)

print("Combinación de LoRAs")
print("\nPodés cargar múltiples LoRAs simultáneamente:")
print("""
# Cargar primer LoRA (LCM)
pipe.load_lora_weights("latent-consistency/lcm-lora-sdxl")

# Cargar segundo LoRA (ej: estilo)
pipe.load_lora_weights(
    "nombre-del-lora-de-estilo",
    adapter_name="style"
)

# Activar ambos con pesos
pipe.set_adapters(["default", "style"], adapter_weights=[1.0, 0.8])
""")

print("\n⚠️ Nota: Los LoRAs deben ser compatibles con el mismo modelo base")
print("Buscá LoRAs en: https://huggingface.co/models?other=lora")

---

## 9. Descargar y Usar LoRAs de la Comunidad

Hay miles de LoRAs gratuitos disponibles en Hugging Face.

In [ ]:
# Ejemplo con un LoRA popular de estilo pixel art
# (Necesitás conexión a internet)

print("Ejemplo: Cargando LoRA de estilo pixel art...\n")

try:
    # Crear nuevo pipeline para no interferir con LCM
    pipe_style = DiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    pipe_style = pipe_style.to(device)

    # Cargar LoRA de estilo (ejemplo)
    # Nota: Reemplazá esto con cualquier LoRA de HuggingFace
    lora_style_id = "nerijs/pixel-art-xl"  # Ejemplo de LoRA público

    pipe_style.load_lora_weights(lora_style_id)

    print(f"LoRA de estilo cargado: {lora_style_id}")

    # Generar con el estilo
    prompt_style = "a medieval castle, pixel art style"

    image_style = pipe_style(
        prompt=prompt_style,
        num_inference_steps=20,
        guidance_scale=7.5,
    ).images[0]

    plt.figure(figsize=(10, 10))
    plt.imshow(image_style)
    plt.axis('off')
    plt.title(f"Con LoRA de Estilo\n{prompt_style}")
    plt.show()

except Exception as e:
    print(f"⚠️ Error cargando LoRA de ejemplo: {e}")
    print("\nEsto es normal si el LoRA específico no está disponible.")
    print("Podés buscar otros LoRAs en: https://huggingface.co/models?other=lora")

---

## 10. Guardar Imágenes con Metadata de LoRA

In [ ]:
from PIL import PngImagePlugin

output_dir = "lcm_lora_generaciones"
os.makedirs(output_dir, exist_ok=True)

def guardar_con_lora_info(image, prompt, lora_name, steps, guidance, output_dir):
    """Guarda imagen con información del LoRA usado"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"lcm_{timestamp}.png"
    filepath = os.path.join(output_dir, filename)

    metadata = PngImagePlugin.PngInfo()
    metadata.add_text("prompt", prompt)
    metadata.add_text("lora", lora_name)
    metadata.add_text("steps", str(steps))
    metadata.add_text("guidance_scale", str(guidance))
    metadata.add_text("model", "SDXL + LCM-LoRA")
    metadata.add_text("timestamp", timestamp)

    image.save(filepath, pnginfo=metadata)

    return filepath

# Generar imagen final
prompt_final = "A mystical forest with glowing mushrooms and fireflies, magical atmosphere, highly detailed"

print(f"Generando imagen final...")
print(f"Prompt: {prompt_final}\n")

imagen_final = pipe(
    prompt=prompt_final,
    num_inference_steps=4,
    guidance_scale=1.0,
    height=1024,
    width=1024,
).images[0]

saved_path = guardar_con_lora_info(
    image=imagen_final,
    prompt=prompt_final,
    lora_name="LCM-LoRA-SDXL",
    steps=4,
    guidance=1.0,
    output_dir=output_dir
)

plt.figure(figsize=(12, 12))
plt.imshow(imagen_final)
plt.axis('off')
plt.title(f"Imagen Final con LCM-LoRA\n{prompt_final}")
plt.tight_layout()
plt.show()

print(f"\nImagen guardada en: {saved_path}")

### ✎ Para pensar

1. Con modelos estándar, `guidance_scale=7.5` es el valor típico. Con LCM, valores mayores a 2.0 degradan la imagen. ¿Qué cambió en el entrenamiento para que esto sea así?
2. ¿Qué efecto produce `guidance_scale=0.0` en LCM? ¿Es equivalente a lo que produce en SDXL-Turbo?

### ✎ Para pensar

1. LCM-LoRA reduce los pasos de 50 a 4-8 sin perder mucha calidad. ¿Qué información tuvo que aprender el LoRA para poder lograr eso?
2. Si ejecutás LCM con 1 solo paso (como Turbo), ¿esperás obtener la misma calidad que SDXL-Turbo? ¿Por qué sí o por qué no?

---

## Ventajas y Limitaciones de LCM-LoRA

### Ventajas
**Universal**: Funciona con múltiples modelos base
**Pequeño**: Solo ~200 MB adicionales
**Rápido**: 5-10x más rápido que generación tradicional
**Plug-and-play**: No requiere reentrenamiento
**Combinable**: Se puede usar con otros LoRAs

### Limitaciones
Guidance scale debe ser bajo (1.0-2.0)
Algunos detalles pueden perderse vs modelos completos
No todos los LoRAs son compatibles entre sí
Requiere scheduler específico (LCMScheduler)

---

## Ejercicios

### Ejercicio 1: Comparación directa
Generá la misma imagen con:
- SDXL sin LoRA (50 pasos)
- SDXL + LCM-LoRA (4 pasos)

Compará calidad y velocidad.

### Ejercicio 2: Optimización de guidance
Para un prompt complejo, probá guidance_scale de 0.5 a 2.5 en pasos de 0.5. ¿Cuál da mejor resultado?

### Ejercicio 3: SD 1.5 vs SDXL
Generá la misma escena con LCM-LoRA en ambos modelos. ¿Qué diferencias notás?

### Ejercicio 4: Búsqueda de LoRAs
Buscá 3 LoRAs de estilo en Hugging Face e intentá cargarlos (pueden necesitar modelos base diferentes).

---

## Para profundizar

**Recursos adicionales:**
- Paper de LCM: https://arxiv.org/abs/2310.04378
- LCM-LoRA SDXL: https://huggingface.co/latent-consistency/lcm-lora-sdxl
- LCM-LoRA SD1.5: https://huggingface.co/latent-consistency/lcm-lora-sdv1-5
- Buscar LoRAs: https://huggingface.co/models?other=lora

**Próximos pasos:**
- Para optimizaciones de memoria: `los materiales de la carpeta `archivo/``
- Para aplicaciones prácticas: `03_Aplicaciones_Practicas_Difusion.ipynb`
- Para crear tus propios LoRAs: Buscar tutoriales de fine-tuning

---

Con esto cerramos la Unidad 9. Dominás las técnicas de generación de imágenes más usadas en la industria actual.

## ⛰️ Lo que construimos

| Concepto | Lo que aprendimos |
|---|---|
| LCM-LoRA | Adapter de ~200 MB que acelera cualquier modelo compatible |
| Pasos de inferencia | 4-8 pasos son suficientes con LCM (vs. 20-50 estándar) |
| guidance_scale | Con LCM el rango óptimo es 1.0-2.0, no 7-15 como en modelos estándar |
| Compatibilidad | Cada LoRA es específico para un modelo base; no son intercambiables |

**Con este cuaderno cerrás la Unidad 9.** Ahora dominás las técnicas de generación de imágenes con modelos de difusión más usadas en la industria.